<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MOSS-SoundEffect-v2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MOSS-SoundEffect-v2_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🎧 MOSS-SoundEffect v2.0 — 48 kHz Flow-Matching DiT SFX & Layered Foley Studio

<p align="left">
  <a href="https://github.com/OpenMOSS/MOSS-TTS/tree/main/moss_soundeffect_v2"><img src="https://img.shields.io/badge/GitHub-OpenMOSS%2FMOSS--TTS-181717?logo=github" alt="GitHub"></a>
  <a href="https://huggingface.co/OpenMOSS-Team/MOSS-SoundEffect-v2.0"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-MOSS--SoundEffect--v2.0-amber" alt="HuggingFace"></a>
  <img src="https://img.shields.io/badge/Sample%20Rate-48%20kHz%20DAC%20VAE-success" alt="48kHz">
  <img src="https://img.shields.io/badge/License-Apache%202.0%20(Ungated)-blue" alt="Apache-2.0">
</p>

Welcome to the **MOSS-SoundEffect v2.0** Google Colab studio! Developed by the **OpenMOSS** team (`OpenMOSS-Team/MOSS-SoundEffect-v2.0`), v2.0 replaces the v1 discrete autoregressive model with a continuous-latent **1.3B Diffusion Transformer (DiT)** trained with **Flow Matching**, paired with a `Qwen3-1.7B` text encoder and a **48 kHz DAC VAE** (`vae_128d_48k.pth`):

- **🎵 Native 48 kHz Crystal-Clear Output** — Generates high-definition 48 kHz sound effects from `1.0s` to `30.0s` with crisp high-frequency transients — completely **ungated under Apache-2.0**.
- **🌐 Bilingual Prompting (English & Chinese)** — Trained natively on both English and Chinese acoustic descriptions (`cfg_scale=4.0`, `sigma_shift=5.0`, `num_inference_steps=100`).
- **🎚️ 3-Layer Composite SFX & UI Sound Mixer** — Synthesize and blend up to **3 independent 48 kHz layers** (*Attack Transient* + *Tonal Body/Chime* + *Resonant Tail/Ambience*) with millisecond offset, gain (`dB`), and stereo panning to craft pro-grade game & OS sound effects.
- **🪟 48 kHz Windows 10/11 Sound Event Pack Studio** — Batch-generate all 16 core Windows `AppEvents\Schemes` sounds at native **48 kHz 16-bit PCM**, complete with zero-latency silence trimming and 1-click `.ps1` + `.reg` installer scripts.

In [ ]:
#@title Step 1: Verify GPU & Install MOSS-SoundEffect v2.0 Dependencies
import os
import sys
import subprocess
import torch
from tqdm.auto import tqdm
from IPython.display import clear_output

REPO_DIR = "/content/MOSS-TTS"
SUBDIR = "/content/MOSS-TTS/moss_soundeffect_v2"

def run_cmd(cmd, check=False):
    print(f"⏳ Running: {cmd}")
    return subprocess.run(cmd, shell=True, check=check)

try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    else:
        print("⚠️ No CUDA GPU detected — GPU runtime is strongly recommended for 1.3B DiT.")

    if not os.path.exists(REPO_DIR):
        run_cmd(f"git clone --depth 1 https://github.com/OpenMOSS/MOSS-TTS.git {REPO_DIR}", check=True)

    run_cmd("apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1", check=False)

    pip_pkgs = (
        "einops safetensors soundfile ftfy huggingface_hub>=0.26.0 "
        "transformers>=4.51.0 diffusers>=0.32.0 accelerate "
        "descript-audiotools descript-audio-codec gradio>=5.0.0 matplotlib scipy librosa torchaudio tqdm"
    )
    run_cmd(f"{sys.executable} -m pip install -q -U {pip_pkgs}", check=False)

    if os.path.exists(SUBDIR):
        run_cmd(f"{sys.executable} -m pip install -q --no-deps -e {SUBDIR}", check=False)
        if SUBDIR not in sys.path:
            sys.path.insert(0, SUBDIR)

    clear_output()
    print("✅ Step 1 Complete: MOSS-SoundEffect v2.0 dependencies installed!")
except Exception as e:
    print(f"❌ Installation Error: {e}")
    raise

In [ ]:
#@title Step 2: Mount Google Drive & Configure Persistent Cache
import os
import sys

WORKSPACE_DIR = "/content/moss_sfx_v2_ws"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.ismount("/content/drive")
except Exception as e:
    print(f"ℹ️ Google Drive not mounted: {e}")

if DRIVE_MOUNTED:
    CACHE_ROOT = "/content/drive/MyDrive/AEI_Audio_Cache/moss_soundeffect_v2"
    OUTPUT_DIR = "/content/drive/MyDrive/AEI_Audio_Exports/MOSS_SoundEffect_v2"
else:
    CACHE_ROOT = os.path.join(WORKSPACE_DIR, "cache")
    OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")

for d in (CACHE_ROOT, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_ROOT, "torch")
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print(f"📂 HF_HOME Cache : {os.environ['HF_HOME']}")
print(f"📂 Output Exports: {OUTPUT_DIR}")

In [ ]:
#@title Step 3: Core MOSS-SoundEffect v2.0 Pipeline, 3-Layer Mixer & Windows Packager
import os
import gc
import sys
import time
import glob
import torch
import numpy as np
import soundfile as sf

SUBDIR = "/content/MOSS-TTS/moss_soundeffect_v2"
if os.path.exists(SUBDIR) and SUBDIR not in sys.path:
    sys.path.insert(0, SUBDIR)

# ==============================================================================
# Shared Windows 10/11 Sound Scheme & Zero-Latency UI Mastering Engine
# ==============================================================================
import io
import re
import json
import uuid
import base64
import shutil
import zipfile
from datetime import datetime
import scipy.signal
import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_SYSTEM_EVENTS = {
    "WindowsLogon": {
        "label": "Windows Logon / Startup",
        "reg_app": ".Default",
        "duration": 3.0,
        "contour": "warm welcoming ascending chord swell with a gentle shimmering reverb tail",
    },
    "WindowsLogoff": {
        "label": "Windows Logoff / Shutdown",
        "reg_app": ".Default",
        "duration": 2.5,
        "contour": "peaceful resolving descending harmonic chord that fades smoothly into silence",
    },
    "WindowsUnlock": {
        "label": "Windows Unlock / Sign-In",
        "reg_app": ".Default",
        "duration": 1.4,
        "contour": "bright welcoming two-note ascending unlock chime with airy resonance",
    },
    "SystemNotification": {
        "label": "System Notification (Toast)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "crisp pleasant two-note notification chime, clear transient attack, short clean tail",
    },
    "WindowsNotifySystemGeneric": {
        "label": "Action Center / Generic Toast Banner",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "modern crystalline two-tone notification ping with soft harmonic decay",
    },
    "SystemAsterisk": {
        "label": "Asterisk / Info Pop-up",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "soft single crystalline ping with subtle harmonic resonance",
    },
    "SystemExclamation": {
        "label": "Exclamation / Warning Alert",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "attention-grabbing mid-range double pulse chime, alert but non-harsh",
    },
    "SystemHand": {
        "label": "Critical Stop / Error",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "low muted percussive double thud with a tense minor interval, error alert",
    },
    "SystemQuestion": {
        "label": "Question / Confirmation Dialog",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "gentle inquisitive rising two-note harmonic inflection chime",
    },
    "Default": {
        "label": "Default Beep / Fallback Alert",
        "reg_app": ".Default",
        "duration": 0.8,
        "contour": "clean short balanced harmonic ping, neutral interface alert",
    },
    "DeviceConnect": {
        "label": "Device Connect (USB Plug-In)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "upbeat two-tone ascending interval (low note to higher fifth), hardware connected",
    },
    "DeviceDisconnect": {
        "label": "Device Disconnect (USB Unplug)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "smooth two-tone descending interval (high note to lower root), hardware unplugged",
    },
    "DeviceFail": {
        "label": "Device Failed to Connect",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "short stuttering muted low buzz double tap, hardware connection failure",
    },
    "EmptyRecycleBin": {
        "label": "Empty Recycle Bin",
        "reg_app": "Explorer",
        "duration": 1.2,
        "contour": "crisp satisfying paper and glass shimmer dissolve whoosh",
    },
    "WindowsUAC": {
        "label": "User Account Control (UAC Prompt)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "authoritative clean two-note security confirmation chime",
    },
    "LowBatteryAlarm": {
        "label": "Low Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "gentle pulsing two-note reminder tone",
    },
    "CriticalBatteryAlarm": {
        "label": "Critical Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.5,
        "contour": "urgent rapid triple pulse warning chime",
    },
    "MailBeep": {
        "label": "New Mail / Message Beep",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "bright airy bell ding notification",
    },
    "MessageNudge": {
        "label": "Message Nudge / Chat Ping",
        "reg_app": ".Default",
        "duration": 0.9,
        "contour": "playful double tactile pop ping notification",
    },
    "PrintComplete": {
        "label": "Print / Task Complete",
        "reg_app": ".Default",
        "duration": 1.3,
        "contour": "satisfying upbeat three-note completion flourish chime",
    },
    "MenuCommand": {
        "label": "Menu Command / Tactile UI Click",
        "reg_app": ".Default",
        "duration": 0.6,
        "contour": "ultra-short crisp tactile UI click pop transient, dry and immediate",
    },
}

SONIC_THEME_PRESETS = {
    "Frutiger Aero Crystal Glass": (
        "crystalline water-drop glass marimba and airy harmonic bell pads, "
        "clean high-tech Frutiger Aero operating system UI sound effect, studio quality"
    ),
    "Cyberpunk Neon HUD": (
        "futuristic cyberpunk holographic HUD telemetry synthesizer, "
        "crisp digital micro-transient with warm sub-bass resonance, sci-fi UI sound"
    ),
    "Cozy Wooden Kalimba & Mallet": (
        "warm organic rosewood marimba, thumb piano kalimba, and soft felt mallet percussion, "
        "intimate close-mic acoustic UI sound effect"
    ),
    "Sci-Fi Spaceship Bridge": (
        "clean starship computer console telemetry bleep, precision FM synthesis harmonic tone, "
        "futuristic spacecraft interface sound effect"
    ),
    "Retro 16-Bit Chiptune Arcade": (
        "crisp nostalgic 16-bit FM synth arcade game UI sound effect, "
        "bright square and triangle wave arpeggio, clean retro console audio"
    ),
    "Warm Lo-Fi Analog Tape Synth": (
        "warm vintage analog poly-synth and Rhodes electric piano chime with gentle tape saturation, "
        "cozy lo-fi interface sound effect"
    ),
    "Minimalist Tactile Micro-Foley": (
        "ultra-clean modern minimalist UI micro-foley, soft mechanical switch click and muted glass tap, "
        "tight dry studio recording"
    ),
    "Dark Fantasy Crystal & Harp": (
        "ethereal celesta, delicate harp harmonic pluck, and mystical resonant crystal chime, "
        "fantasy RPG interface sound effect"
    ),
}

DEFAULT_SELECTED_WIN_EVENTS = [
    "WindowsLogon",
    "WindowsLogoff",
    "WindowsUnlock",
    "SystemNotification",
    "WindowsNotifySystemGeneric",
    "SystemAsterisk",
    "SystemExclamation",
    "SystemHand",
    "DeviceConnect",
    "DeviceDisconnect",
    "EmptyRecycleBin",
    "WindowsUAC",
]


def build_windows_event_prompt(event_key: str, theme_name: str, custom_style: str = "") -> str:
    ev = WINDOWS_SYSTEM_EVENTS.get(event_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
    style_desc = custom_style.strip() if custom_style and custom_style.strip() else SONIC_THEME_PRESETS.get(
        theme_name, SONIC_THEME_PRESETS["Frutiger Aero Crystal Glass"]
    )
    return f"{ev['contour']}, {style_desc}, high fidelity, no background noise"


def apply_phase_safe_stereo_widener(arr: np.ndarray, sr: int, width: float = 0.22) -> np.ndarray:
    """Upgrade mono (N, 1) to phase-coherent stereo (N, 2) or subtly widen stereo above 350 Hz."""
    if arr.ndim == 1:
        arr = arr[:, None]
    n_samples, ch = arr.shape
    if n_samples < 64 or width <= 0.01:
        return np.repeat(arr, 2, axis=1) if ch == 1 else arr

    nyq = 0.5 * float(sr)
    cutoff = min(max(350.0 / nyq, 0.01), 0.85)
    b, a = scipy.signal.butter(2, cutoff, btype="highpass")

    if ch == 1:
        mono = arr[:, 0]
        high = scipy.signal.lfilter(b, a, mono).astype(np.float32)
        delay_samples = max(1, int(0.00038 * sr))  # ~0.38 ms Haas micro-decorrelation
        side = np.zeros_like(high)
        if delay_samples < n_samples:
            side[delay_samples:] = high[:-delay_samples] - high[delay_samples:]
        side *= float(width)
        left = mono + side
        right = mono - side
        return np.stack([left, right], axis=1).astype(np.float32)
    else:
        mid = 0.5 * (arr[:, 0] + arr[:, 1])
        side = 0.5 * (arr[:, 0] - arr[:, 1])
        side_high = scipy.signal.lfilter(b, a, side).astype(np.float32)
        side_boosted = side + float(width) * 0.5 * side_high
        return np.stack([mid + side_boosted, mid - side_boosted], axis=1).astype(np.float32)


def master_ui_sound(
    audio_np: np.ndarray,
    sr: int,
    target_duration_s: float = None,
    trim_silence: bool = True,
    trim_db: float = -45.0,
    fade_in_ms: float = 1.5,
    fade_out_ms: float = 120.0,
    target_dbfs: float = -1.2,
    stereo_widen: bool = True,
    hpf_hz: float = 40.0,
    pitch_semitones: float = 0.0,
) -> np.ndarray:
    """Zero-latency UI sound mastering: snaps to first transient (<1ms pre-roll), 40Hz sub-rumble filter, optional semitone transpose, 1.5ms click-free micro-fade, phase-safe stereo widening, and true-peak normalization."""
    arr = np.asarray(audio_np, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[:, None]  # (samples, 1)
    elif arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
        arr = arr.T  # (samples, channels)

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = arr - np.mean(arr, axis=0, keepdims=True)

    # Optional sub-rumble high-pass filter (default 40 Hz) to keep UI chimes crisp
    if hpf_hz and float(hpf_hz) > 10.0 and arr.shape[0] > 64:
        nyq = 0.5 * float(sr)
        cutoff = min(max(float(hpf_hz) / nyq, 0.002), 0.45)
        b_hp, a_hp = scipy.signal.butter(2, cutoff, btype="highpass")
        arr = scipy.signal.lfilter(b_hp, a_hp, arr, axis=0).astype(np.float32)

    # Optional semitone pitch shift via high-precision polyphase resampling
    if pitch_semitones and abs(float(pitch_semitones)) >= 0.05 and arr.shape[0] > 64:
        factor = 2.0 ** (float(pitch_semitones) / 12.0)
        up = 1000
        down = max(100, int(round(1000.0 * factor)))
        gcd = np.gcd(up, down)
        arr = scipy.signal.resample_poly(arr, up // gcd, down // gcd, axis=0).astype(np.float32)

    if trim_silence and arr.shape[0] > int(0.04 * sr):
        mono_env = np.max(np.abs(arr), axis=1)
        diff_env = np.abs(np.diff(mono_env, prepend=mono_env[0]))
        peak = float(np.max(mono_env))
        if peak > 1e-6:
            amp_thresh = peak * (10.0 ** (float(trim_db) / 20.0))
            flux_thresh = float(np.max(diff_env)) * 0.08
            active_idx = np.where((mono_env >= amp_thresh) | (diff_env >= flux_thresh))[0]
            if active_idx.size > 0:
                pre_roll = int(0.0008 * sr)   # 0.8 ms zero-latency pre-roll
                post_roll = int(0.160 * sr)   # 160 ms natural reverb tail retention
                start_i = max(0, int(active_idx[0]) - pre_roll)
                end_i = min(arr.shape[0], int(active_idx[-1]) + post_roll)
                if end_i > start_i + int(0.03 * sr):
                    arr = arr[start_i:end_i]

    if target_duration_s is not None and target_duration_s > 0:
        max_samples = int(float(target_duration_s) * sr)
        if arr.shape[0] > max_samples:
            arr = arr[:max_samples]

    n_samples = arr.shape[0]
    if n_samples == 0:
        return np.zeros((int(0.5 * sr), 2), dtype=np.float32)

    if stereo_widen:
        arr = apply_phase_safe_stereo_widener(arr, sr, width=0.22 if arr.shape[1] == 1 else 0.12)

    fi_samples = min(int((float(fade_in_ms) / 1000.0) * sr), max(1, n_samples // 8))
    if fi_samples > 1:
        fi_curve = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, fi_samples, dtype=np.float32)))
        arr[:fi_samples] *= fi_curve[:, None]

    fo_samples = min(int((float(fade_out_ms) / 1000.0) * sr), max(1, n_samples // 2))
    if fo_samples > 1:
        fo_curve = (0.5 * (1.0 + np.cos(np.linspace(0.0, np.pi, fo_samples, dtype=np.float32)))) ** 1.2
        arr[-fo_samples:] *= fo_curve[:, None]

    peak_val = float(np.max(np.abs(arr)))
    if peak_val > 1e-6:
        target_amp = 10.0 ** (float(target_dbfs) / 20.0)
        arr = arr * (target_amp / peak_val)

    arr = np.clip(arr, -0.999, 0.999)
    return arr if arr.shape[1] > 1 else arr[:, 0]


def save_pcm16_wav(path: str, audio_np: np.ndarray, sr: int, subtype: str = "PCM_16") -> str:
    abs_path = os.path.abspath(path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    sf.write(abs_path, audio_np, int(sr), subtype=subtype)
    # Mirror to AEI_TTS_Out if Google Drive is mounted so companion notebooks (InfiniteTalk, SeedVR2, Audio_PostProcessor) auto-discover it
    if "/content/drive/MyDrive/" in abs_path and "/AEI_TTS_Out/" not in abs_path:
        try:
            mirror_dir = "/content/drive/MyDrive/AEI_TTS_Out/SFX_Exports"
            os.makedirs(mirror_dir, exist_ok=True)
            shutil.copy2(abs_path, os.path.join(mirror_dir, os.path.basename(abs_path)))
        except Exception:
            pass
    return abs_path


def load_audio_file_safe(audio_path: str, target_sr: int = None, mono: bool = False):
    """Robust audio loader using soundfile first, falling back to librosa/torchaudio."""
    try:
        data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
        # data is (T, C) -> transpose to (C, T)
        wav = data.T
    except Exception:
        import librosa
        wav, sr = librosa.load(audio_path, sr=None, mono=False)
        if wav.ndim == 1:
            wav = wav[None, :]
    if mono and wav.shape[0] > 1:
        wav = np.mean(wav, axis=0, keepdims=True)
    if target_sr is not None and int(sr) != int(target_sr):
        gcd = np.gcd(int(sr), int(target_sr))
        wav = scipy.signal.resample_poly(wav, int(target_sr) // gcd, int(sr) // gcd, axis=-1).astype(np.float32)
        sr = int(target_sr)
    return wav, int(sr)


def render_waveform_and_spectrogram(wav_path: str, title: str = "Audio Signal & Spectrogram") -> str:
    """Render a dual-panel waveform + spectrogram plot for visual transient & tail inspection."""
    try:
        data, sr = sf.read(wav_path, always_2d=True)
        mono = np.mean(data, axis=1)
        dur = len(mono) / float(sr)
        t = np.linspace(0.0, dur, len(mono))

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 4.2), facecolor="#0f172a")
        for ax in (ax1, ax2):
            ax.set_facecolor("#1e293b")
            ax.tick_params(colors="#cbd5e1", labelsize=8)
            for spine in ax.spines.values():
                spine.set_color("#334155")

        if data.shape[1] > 1:
            ax1.plot(t, data[:, 0], color="#38bdf8", linewidth=0.8, alpha=0.85, label="L")
            ax1.plot(t, data[:, 1], color="#f472b6", linewidth=0.8, alpha=0.75, label="R")
            ax1.legend(loc="upper right", fontsize=7, facecolor="#0f172a", edgecolor="#334155", labelcolor="#f8fafc")
        else:
            ax1.plot(t, mono, color="#38bdf8", linewidth=0.9)
        ax1.set_xlim(0, max(dur, 0.05))
        ax1.set_ylim(-1.05, 1.05)
        ax1.set_ylabel("Amplitude", color="#e2e8f0", fontsize=8)
        ch_label = "Stereo" if data.shape[1] > 1 else "Mono"
        ax1.set_title(f"{title} ({sr} Hz {ch_label}, {dur:.2f}s)", color="#f8fafc", fontsize=10, fontweight="bold")
        ax1.grid(True, color="#334155", alpha=0.4, linestyle="--")

        nfft = min(1024, max(128, 2 ** int(np.log2(max(len(mono) // 8, 128)))))
        ax2.specgram(mono + 1e-10, NFFT=nfft, Fs=sr, noverlap=nfft // 2, cmap="magma")
        ax2.set_xlim(0, max(dur, 0.05))
        ax2.set_xlabel("Time (seconds)", color="#e2e8f0", fontsize=8)
        ax2.set_ylabel("Freq (Hz)", color="#e2e8f0", fontsize=8)

        plt.tight_layout()
        out_png = os.path.join("/tmp", f"spec_{uuid.uuid4().hex[:8]}.png")
        fig.savefig(out_png, dpi=120, facecolor=fig.get_facecolor())
        plt.close(fig)
        return out_png
    except Exception as e:
        print(f"Spectrogram render warning: {e}")
        return None


def build_audition_soundboard_html(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    embed_base64: bool = False,
) -> str:
    """Build a dark-mode HTML5 WebAudio interactive soundboard with 1-click sequential Play-All auditioning."""
    board_uid = f"sb_{uuid.uuid4().hex[:8]}"
    cards_html = []
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"label": ev_key, "reg_app": ".Default", "duration": 1.0})
        if embed_base64 and os.path.exists(wav_info["path"]):
            with open(wav_info["path"], "rb") as bf:
                b64 = base64.b64encode(bf.read()).decode("ascii")
            audio_src = f"data:audio/wav;base64,{b64}"
        else:
            audio_src = f"Media/{wav_info['filename']}"

        cards_html.append(
            f'<div class="aei-sb-card" style="background:#1e293b;border:1px solid #334155;border-radius:10px;padding:10px 12px;display:flex;flex-direction:column;gap:6px;transition:all 0.2s ease;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;">'
            f'<strong style="color:#f8fafc;font-size:13px;">🔔 {ev_meta.get("label", ev_key)}</strong>'
            f'<span style="background:#0f172a;color:#38bdf8;font-size:11px;padding:2px 6px;border-radius:4px;font-family:monospace;">{ev_meta.get("reg_app", ".Default")}\\{ev_key}</span>'
            f'</div>'
            f'<audio controls preload="none" src="{audio_src}" style="width:100%;height:32px;"></audio>'
            f'</div>'
        )
    grid_str = "\n".join(cards_html)
    play_all_js = (
        f"const root=document.getElementById('{board_uid}');"
        f"if(!root)return;"
        f"const audios=Array.from(root.querySelectorAll('audio'));"
        f"const cards=Array.from(root.querySelectorAll('.aei-sb-card'));"
        f"if(root._playing){{root._playing=false;audios.forEach(a=>{{a.pause();a.currentTime=0;}});cards.forEach(c=>c.style.borderColor='#334155');this.textContent='▶ Play All Events Sequentially';return;}}"
        f"root._playing=true;this.textContent='⏹ Stop Sequence';"
        f"let idx=0;const btn=this;"
        f"const next=()=>{{if(!root._playing||idx>=audios.length){{root._playing=false;btn.textContent='▶ Play All Events Sequentially';cards.forEach(c=>c.style.borderColor='#334155');return;}}"
        f"cards.forEach((c,i)=>{{c.style.borderColor=(i===idx)?'#38bdf8':'#334155';}});"
        f"const a=audios[idx++];a.currentTime=0;"
        f"a.onended=()=>{{setTimeout(next,280);}};"
        f"a.onerror=()=>{{setTimeout(next,100);}};"
        f"a.play().catch(()=>setTimeout(next,100));}};"
        f"next();"
    )
    return (
        f'<div id="{board_uid}" style="background:#0f172a;color:#f8fafc;padding:16px;border-radius:12px;border:1px solid #334155;font-family:system-ui,-apple-system,sans-serif;">'
        f'<div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;margin-bottom:12px;">'
        f'<div>'
        f'<h3 style="margin:0 0 4px 0;color:#38bdf8;font-size:16px;">🪟 {scheme_name} — Interactive Windows 10/11 Soundboard</h3>'
        f'<p style="margin:0;color:#94a3b8;font-size:12px;">Theme: <b>{theme_name}</b> · Engine: <b>{model_label}</b> · Events: <b>{len(event_wav_map)}</b></p>'
        f'</div>'
        f'<button type="button" onclick="{play_all_js}" style="background:#0284c7;color:#ffffff;border:none;border-radius:8px;padding:8px 14px;font-size:12px;font-weight:600;cursor:pointer;">▶ Play All Events Sequentially</button>'
        f'</div>'
        f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(270px,1fr));gap:10px;">'
        f'{grid_str}'
        f'</div>'
        f'</div>'
    )


LAST_WINPACK_STATE = {
    "pack_dir": None,
    "scheme_name": "AEI_Custom_Scheme",
    "theme_name": "Frutiger Aero Crystal Glass",
    "model_label": "AEI Sound Studio",
    "output_dir": "/content",
    "event_wav_map": {},
}


def package_windows_sound_scheme_zip(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    output_dir: str,
) -> str:
    """Create a complete Windows 10/11 Sound Scheme ZIP with .wav files, Install/Uninstall .ps1, .reg file, Scheme.ini, soundpack_manifest.json, and audition_board.html."""
    safe_id = re.sub(r"[^A-Za-z0-9_-]+", "_", (scheme_name or "AEI_Custom_Sounds").strip())[:32].strip("_") or "AEI_Custom"
    display_name = (scheme_name or "AEI Custom Sounds").strip()[:48]
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = os.path.join(output_dir, f"{safe_id}_Windows_SoundPack_{ts}.zip")

    ps1_lines = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Automated Installer: {display_name}",
        f"# Generated by AEI Colab Notebooks ({model_label})",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'Stop'",
        f"$SchemeId = '{safe_id}'",
        f"$SchemeDisplay = '{display_name}'",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"New-Item -ItemType Directory -Force -Path $TargetDir | Out-Null",
        f"$BackupReg = Join-Path $TargetDir 'Backup_Previous_AppEvents.reg'",
        f"if (-not (Test-Path $BackupReg)) {{ reg export 'HKCU\\AppEvents\\Schemes' $BackupReg /y | Out-Null }}",
        f"Copy-Item -Path (Join-Path $PSScriptRoot 'Media\\*.wav') -Destination $TargetDir -Force",
        f"",
        f"$NamesKey = \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\"",
        f"New-Item -Path $NamesKey -Force | Out-Null",
        f"Set-ItemProperty -Path $NamesKey -Name '(Default)' -Value $SchemeDisplay",
        f"",
        f"$EventMap = @{{",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        ps1_lines.append(f"    '{ev_meta['reg_app']}\\{ev_key}' = '{wav_info['filename']}'")
    ps1_lines += [
        f"}}",
        f"$FirstWav = $null",
        f"foreach ($entry in $EventMap.GetEnumerator()) {{",
        f"    $wavPath = Join-Path $TargetDir $entry.Value",
        f"    if (-not $FirstWav -and (Test-Path $wavPath)) {{ $FirstWav = $wavPath }}",
        f"    $schemeEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\$SchemeId\"",
        f"    $currentEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\.Current\"",
        f"    New-Item -Path $schemeEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $schemeEventKey -Name '(Default)' -Value $wavPath",
        f"    New-Item -Path $currentEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $currentEventKey -Name '(Default)' -Value $wavPath",
        f"}}",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value $SchemeId",
        f"if ($FirstWav) {{ try {{ (New-Object System.Media.SoundPlayer $FirstWav).Play() }} catch {{}} }}",
        f"Write-Host \"Successfully installed and activated Windows Sound Scheme: $SchemeDisplay\" -ForegroundColor Green",
        f"Write-Host \"Prior registry state backed up to: $BackupReg\" -ForegroundColor Cyan",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    uninstall_ps1 = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Uninstaller & Default Restore: {display_name}",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'SilentlyContinue'",
        f"$SchemeId = '{safe_id}'",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value '.Default'",
        f"Get-ChildItem 'HKCU:\\AppEvents\\Schemes\\Apps' -Recurse | Where-Object {{ $_.PSChildName -eq $SchemeId }} | Remove-Item -Recurse -Force",
        f"Remove-Item -Path \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\" -Recurse -Force",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"if (Test-Path $TargetDir) {{ Remove-Item -Path $TargetDir -Recurse -Force }}",
        f"Write-Host \"Restored Windows Default sound scheme and removed $SchemeId.\" -ForegroundColor Yellow",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    reg_lines = [
        "Windows Registry Editor Version 5.00",
        "",
        f"; Windows Sound Scheme: {display_name} ({model_label})",
        f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Names\\{safe_id}]",
        f"@=\"{display_name}\"",
        "",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        escaped_path = f"C:\\\\Windows\\\\Media\\\\{safe_id}\\\\{wav_info['filename']}"
        reg_lines += [
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\{safe_id}]",
            f"@=\"{escaped_path}\"",
            "",
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\.Current]",
            f"@=\"{escaped_path}\"",
            "",
        ]

    ini_lines = [
        "[Scheme]",
        f"Name={display_name}",
        f"SchemeId={safe_id}",
        f"Theme={theme_name}",
        f"Generator={model_label}",
        f"CreatedAt={datetime.now().isoformat()}",
        "",
        "[Events]",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ini_lines.append(f"{ev_key}=Media/{wav_info['filename']}")
        ini_lines.append(f"{ev_key}_Prompt={wav_info.get('prompt', '')}")

    manifest_data = {
        "scheme_name": display_name,
        "scheme_id": safe_id,
        "theme_name": theme_name,
        "generator": model_label,
        "created_at": datetime.now().isoformat(),
        "events": {
            k: {
                "filename": f"Media/{v['filename']}",
                "label": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("label", k),
                "reg_app": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("reg_app", ".Default"),
                "prompt": v.get("prompt", ""),
            }
            for k, v in event_wav_map.items()
        },
    }

    standalone_html = (
        "<!DOCTYPE html>\n<html><head><meta charset='utf-8'>"
        f"<title>{display_name} — Windows Soundboard</title></head>"
        "<body style='background:#020617;margin:24px;'>"
        + build_audition_soundboard_html(display_name, event_wav_map, theme_name, model_label, embed_base64=False)
        + "</body></html>"
    )

    readme_txt = (
        f"====================================================================\n"
        f"  {display_name} — Windows 10 / 11 Sound Event Pack\n"
        f"  Generated with: {model_label}\n"
        f"  Theme Preset:   {theme_name}\n"
        f"====================================================================\n\n"
        f"PREVIEW IN BROWSER FIRST:\n"
        f"  Open 'audition_board.html' in any browser to audition all {len(event_wav_map)} events!\n\n"
        f"OPTION 1: 1-Click PowerShell Install (Recommended, No Admin Required)\n"
        f"--------------------------------------------------------------------\n"
        f"1. Extract this entire ZIP archive to any folder.\n"
        f"2. Right-click 'Install_{safe_id}.ps1' -> 'Run with PowerShell'.\n"
        f"3. The script copies the .wav files to %LOCALAPPDATA%\\WindowsSoundSchemes\\{safe_id},\n"
        f"   registers '{display_name}' in the Windows Sound Control Panel, activates it,\n"
        f"   and opens mmsys.cpl so you can preview or customize individual events!\n"
        f"4. To revert at any time, right-click 'Uninstall_{safe_id}.ps1' -> 'Run with PowerShell'.\n\n"
        f"OPTION 2: Manual Registry (.reg) Import\n"
        f"--------------------------------------------------------------------\n"
        f"1. Copy the 'Media' folder contents into 'C:\\Windows\\Media\\{safe_id}\\'.\n"
        f"2. Double-click 'Install_{safe_id}.reg' to register the sound scheme.\n"
    )

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for ev_key, wav_info in event_wav_map.items():
            zf.write(wav_info["path"], arcname=f"Media/{wav_info['filename']}")
        zf.writestr(f"Install_{safe_id}.ps1", "\r\n".join(ps1_lines))
        zf.writestr(f"Uninstall_{safe_id}.ps1", "\r\n".join(uninstall_ps1))
        zf.writestr(f"Install_{safe_id}.reg", "\r\n".join(reg_lines))
        zf.writestr("Scheme.ini", "\r\n".join(ini_lines))
        zf.writestr("soundpack_manifest.json", json.dumps(manifest_data, indent=2))
        zf.writestr("audition_board.html", standalone_html)
        zf.writestr("README_INSTALL.txt", readme_txt.replace("\n", "\r\n"))

    if event_wav_map:
        first_entry = next(iter(event_wav_map.values()))
        if isinstance(first_entry, dict) and "path" in first_entry:
            LAST_WINPACK_STATE["pack_dir"] = os.path.dirname(first_entry["path"])
    LAST_WINPACK_STATE["scheme_name"] = display_name
    LAST_WINPACK_STATE["theme_name"] = theme_name
    LAST_WINPACK_STATE["model_label"] = model_label
    LAST_WINPACK_STATE["output_dir"] = output_dir
    LAST_WINPACK_STATE["event_wav_map"] = dict(event_wav_map)
    return zip_path


def finalize_rerolled_event_in_pack(
    ev_key: str,
    mastered_wav: np.ndarray,
    sr: int,
    ev_prompt: str,
    output_dir: str,
    scheme_name: str = "",
    theme_name: str = "",
    model_label: str = "",
):
    """Save a re-rolled single Windows event .wav into the active pack state, re-bundle the ZIP, and refresh the soundboard."""
    pack_dir = LAST_WINPACK_STATE.get("pack_dir")
    if not pack_dir or not os.path.exists(pack_dir):
        pack_dir = os.path.join(output_dir, f"winpack_reroll_{int(datetime.now().timestamp())}")
        os.makedirs(pack_dir, exist_ok=True)
        LAST_WINPACK_STATE["pack_dir"] = pack_dir

    fname = f"{ev_key}.wav"
    fpath = os.path.join(pack_dir, fname)
    save_pcm16_wav(fpath, mastered_wav, sr)

    ev_map = dict(LAST_WINPACK_STATE.get("event_wav_map") or {})
    ev_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
    LAST_WINPACK_STATE["event_wav_map"] = ev_map

    s_name = (scheme_name or LAST_WINPACK_STATE.get("scheme_name") or "AEI_Custom_Scheme").strip()
    t_name = (theme_name or LAST_WINPACK_STATE.get("theme_name") or "Custom Theme").strip()
    m_lbl = (model_label or LAST_WINPACK_STATE.get("model_label") or "AEI Sound Studio").strip()

    zip_path = package_windows_sound_scheme_zip(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        output_dir=output_dir,
    )
    LAST_WINPACK_STATE["pack_dir"] = pack_dir
    board_html = build_audition_soundboard_html(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        embed_base64=True,
    )
    spec_img = render_waveform_and_spectrogram(fpath, title=f"Re-Rolled Event: {ev_key}")
    status = (
        f"🔄 Re-rolled **`{ev_key}`** ({WINDOWS_SYSTEM_EVENTS.get(ev_key, {}).get('label', ev_key)}) and updated "
        f"**`{os.path.basename(zip_path)}`** ({len(ev_map)} total events in pack)!"
    )
    return zip_path, fpath, spec_img, board_html, status


MOSS_PRESETS = {
    "🪟 Glass Marimba Notification (EN)": "A crisp crystalline glass marimba two-note ascending notification chime with a soft airy tail.",
    "🪟 清脆水晶通知音 (ZH)": "清脆悦耳的双音符水晶马林巴琴系统通知提示音，带有柔和空灵的尾音。",
    "🔌 Futuristic Sci-Fi USB Connect (EN)": "Futuristic sci-fi hologram interface telemetry ping, upbeat ascending two-tone synth chime.",
    "🔌 科幻全息设备连接音 (ZH)": "未来感科幻全息界面连接音效，明亮上扬的双音调电子合成器提示音。",
    "⚔️ Heavy Sword Draw & Metallic Ring (EN)": "A sharp metallic sword being drawn from a leather scabbard with a bright resonant steel ring.",
    "🌧️ Rain on Window & Distant Thunder (EN)": "Gentle rain pattering against a glass window pane followed by a low distant rumble of thunder.",
    "👣 Footsteps on Gravel Path (EN)": "Crisp close-up foley of footsteps walking steadily on loose dry gravel outdoors.",
    "🤖 Mecha Hydraulic Step & Servo (EN)": "Heavy robotic mecha footstep impact followed by high-tech hydraulic servo motor whirring.",
}


class MossSoundEffectV2Studio:
    """Wrapper for OpenMOSS-Team/MOSS-SoundEffect-v2.0 (48 kHz 1.3B Flow-Matching DiT)."""

    def __init__(self, model_id="OpenMOSS-Team/MOSS-SoundEffect-v2.0"):
        self.model_id = model_id
        self.pipe = None
        self.sample_rate = 48000
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def unload(self):
        self.pipe = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def ensure_loaded(self):
        if self.pipe is not None:
            return "✅ Using cached `MOSS-SoundEffect-v2.0` (48 kHz Flow-Matching DiT)."
        from moss_soundeffect_v2 import MossSoundEffectPipeline
        dtype = torch.bfloat16 if (self.device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
        self.pipe = MossSoundEffectPipeline.from_pretrained(
            self.model_id,
            torch_dtype=dtype if self.device == "cuda" else torch.float32,
            device=self.device,
        )
        return f"✅ Loaded `{self.model_id}` in `{dtype}` on `{self.device}`."

    @torch.inference_mode()
    def generate_one(
        self,
        prompt: str,
        negative_prompt: str = "",
        seconds: float = 5.0,
        num_inference_steps: int = 100,
        cfg_scale: float = 4.0,
        sigma_shift: float = 5.0,
        seed: int = -1,
    ):
        msg = self.ensure_loaded()
        actual_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))
        torch.manual_seed(actual_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(actual_seed)

        call_kwargs = dict(
            prompt=prompt.strip(),
            seconds=float(seconds),
            num_inference_steps=int(num_inference_steps),
            cfg_scale=float(cfg_scale),
        )
        import inspect
        sig = inspect.signature(self.pipe.__call__)
        if "negative_prompt" in sig.parameters and negative_prompt and negative_prompt.strip():
            call_kwargs["negative_prompt"] = negative_prompt.strip()
        if "max_inference_seconds" in sig.parameters:
            # Speedup: avoid denoising a full 30s latent when generating short 1-4s UI chimes
            call_kwargs["max_inference_seconds"] = max(5, min(30, int(np.ceil(float(seconds) + 1.0))))
        if "sigma_shift" in sig.parameters:
            call_kwargs["sigma_shift"] = float(sigma_shift)
        if "seed" in sig.parameters:
            call_kwargs["seed"] = actual_seed

        audio_tensor = self.pipe(**call_kwargs)
        if isinstance(audio_tensor, (tuple, list)):
            audio_tensor = audio_tensor[0]
        arr = audio_tensor.detach().cpu().float().numpy()
        arr = np.squeeze(arr)
        if arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
            arr = arr.T
        return arr, self.sample_rate, actual_seed, msg


def mix_stereo_layers(layers_info: list, sr: int = 48000) -> np.ndarray:
    """Mix multiple mono/stereo (waveform, offset_ms, gain_db, pan) layers into a 48 kHz stereo waveform."""
    max_len = 0
    processed = []
    for wav, offset_ms, gain_db, pan in layers_info:
        if wav is None:
            continue
        w = np.asarray(wav, dtype=np.float32)
        if w.ndim == 2:
            w = np.mean(w, axis=1)
        offset_samples = max(0, int((float(offset_ms) / 1000.0) * sr))
        gain = 10.0 ** (float(gain_db) / 20.0)
        angle = (np.clip(float(pan), -1.0, 1.0) + 1.0) * (np.pi / 4.0)
        l_gain = gain * np.cos(angle)
        r_gain = gain * np.sin(angle)
        total_len = offset_samples + len(w)
        max_len = max(max_len, total_len)
        processed.append((w, offset_samples, l_gain, r_gain))

    if max_len == 0 or not processed:
        return np.zeros((sr, 2), dtype=np.float32)

    stereo = np.zeros((max_len, 2), dtype=np.float32)
    for w, off, lg, rg in processed:
        stereo[off : off + len(w), 0] += w * lg
        stereo[off : off + len(w), 1] += w * rg

    peak = float(np.max(np.abs(stereo)))
    if peak > 1e-6:
        stereo = stereo / peak * 0.90
    return stereo


MOSS_STUDIO = MossSoundEffectV2Studio()
print("✅ Step 3 Complete: MOSS-SoundEffect v2.0 48 kHz engine initialized!")

In [ ]:
#@title Step 4: Launch MOSS-SoundEffect v2.0 Gradio Studio (4 Tabs)
import os
import time
import gradio as gr
from IPython.display import clear_output


def ui_moss_generate(
    prompt,
    negative_prompt,
    seconds,
    steps,
    cfg_scale,
    sigma_shift,
    seed,
    num_variations,
    apply_mastering,
    trim_db,
    fade_out_ms,
):
    try:
        if not prompt or not prompt.strip():
            return None, None, None, "❌ Please enter an English or Chinese sound effect prompt."
        t0 = time.time()
        base_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))
        saved = []
        load_msg = ""
        for i in range(int(num_variations)):
            arr, sr, used_seed, load_msg = MOSS_STUDIO.generate_one(
                prompt=prompt,
                negative_prompt=negative_prompt,
                seconds=float(seconds),
                num_inference_steps=int(steps),
                cfg_scale=float(cfg_scale),
                sigma_shift=float(sigma_shift),
                seed=base_seed + i,
            )
            if apply_mastering:
                arr = master_ui_sound(
                    arr, sr, target_duration_s=float(seconds), trim_silence=True,
                    trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
                )
            out_path = os.path.join(OUTPUT_DIR, f"moss_v2_48k_{int(time.time())}_s{used_seed}_v{i+1}.wav")
            save_pcm16_wav(out_path, arr, sr)
            saved.append(out_path)

        spec = render_waveform_and_spectrogram(saved[0], title="MOSS-SoundEffect v2.0 (48 kHz)")
        second_wav = saved[1] if len(saved) > 1 else None
        status = (
            f"✅ Synthesized **{len(saved)}** 48 kHz `.wav` clip(s) in **{time.time()-t0:.2f}s** (Base Seed: `{base_seed}`).\n"
            f"- {load_msg}\n- Saved to: `{saved[0]}`"
        )
        return saved[0], second_wav, spec, status
    except Exception as e:
        return None, None, None, f"❌ MOSS-SoundEffect v2.0 Error: {e}"


def ui_moss_layer_mixer(
    p1, dur1, off1, gain1, pan1,
    p2, dur2, off2, gain2, pan2,
    p3, dur3, off3, gain3, pan3,
    steps, cfg_scale, seed,
):
    try:
        t0 = time.time()
        base_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(100, 900000))
        layers = []
        layer_files = [None, None, None]
        specs = [(p1, dur1, off1, gain1, pan1), (p2, dur2, off2, gain2, pan2), (p3, dur3, off3, gain3, pan3)]
        sr = 48000

        for idx, (p, d, off, g, pan) in enumerate(specs):
            if p and p.strip():
                arr, sr, _, _ = MOSS_STUDIO.generate_one(
                    prompt=p.strip(), seconds=float(d), num_inference_steps=int(steps),
                    cfg_scale=float(cfg_scale), seed=base_seed + idx * 17,
                )
                arr = master_ui_sound(arr, sr, target_duration_s=float(d), trim_silence=True)
                l_path = os.path.join(OUTPUT_DIR, f"moss_layer{idx+1}_{int(time.time())}.wav")
                save_pcm16_wav(l_path, arr, sr)
                layer_files[idx] = l_path
                layers.append((arr, off, g, pan))

        if not layers:
            return None, None, None, None, None, "❌ Enter at least one layer prompt."

        mixed_stereo = mix_stereo_layers(layers, sr=sr)
        mixed_stereo = master_ui_sound(mixed_stereo, sr, trim_silence=True, fade_in_ms=5.0, fade_out_ms=150.0)
        mix_path = os.path.join(OUTPUT_DIR, f"moss_composite_stereo_48k_{int(time.time())}.wav")
        save_pcm16_wav(mix_path, mixed_stereo, sr)
        spec_img = render_waveform_and_spectrogram(mix_path, title="3-Layer Composite 48 kHz Stereo Mix")
        msg = f"✅ Synthesized and mixed **{len(layers)}** 48 kHz layers into stereo `{mix_path}` in **{time.time()-t0:.1f}s**!"
        return mix_path, layer_files[0], layer_files[1], layer_files[2], spec_img, msg
    except Exception as e:
        return None, None, None, None, None, f"❌ Layer Mixer Error: {e}"


def ui_moss_windows_pack(
    scheme_name,
    theme_name,
    custom_style,
    selected_events,
    steps,
    cfg_scale,
    sigma_shift,
    base_seed,
    trim_db,
    fade_out_ms,
):
    try:
        if not selected_events:
            return None, None, None, "", "❌ Select at least one Windows system event."
        t0 = time.time()
        pack_dir = os.path.join(OUTPUT_DIR, f"moss_winpack_{int(time.time())}")
        os.makedirs(pack_dir, exist_ok=True)
        event_wav_map = {}
        first_wav = None
        seed_val = int(base_seed) if int(base_seed) >= 0 else int(np.random.randint(1000, 900000))

        for idx, ev_key in enumerate(selected_events):
            ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
            ev_prompt = build_windows_event_prompt(ev_key, theme_name, custom_style)
            dur = float(ev_meta["duration"])
            arr, sr, _, _ = MOSS_STUDIO.generate_one(
                prompt=ev_prompt,
                seconds=max(dur + 0.5, 1.5),
                num_inference_steps=int(steps),
                cfg_scale=float(cfg_scale),
                sigma_shift=float(sigma_shift),
                seed=seed_val + idx,
            )
            mastered = master_ui_sound(
                arr, sr, target_duration_s=dur, trim_silence=True,
                trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
            fname = f"{ev_key}.wav"
            fpath = os.path.join(pack_dir, fname)
            save_pcm16_wav(fpath, mastered, sr)
            event_wav_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
            if first_wav is None:
                first_wav = fpath

        zip_path = package_windows_sound_scheme_zip(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=theme_name,
            model_label="MOSS-SoundEffect-v2.0 (48 kHz)",
            output_dir=OUTPUT_DIR,
        )
        board_html = build_audition_soundboard_html(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=theme_name,
            model_label="MOSS-SoundEffect-v2.0 (48 kHz)",
            embed_base64=True,
        )
        spec_img = render_waveform_and_spectrogram(first_wav, title=f"48 kHz Windows Event: {selected_events[0]}") if first_wav else None
        status = (
            f"✅ Packaged **{len(event_wav_map)}** native **48 kHz 16-bit PCM** Windows events into "
            f"`{os.path.basename(zip_path)}` in **{time.time()-t0:.1f}s**!"
        )
        return zip_path, first_wav, spec_img, board_html, status
    except Exception as e:
        return None, None, None, "", f"❌ Windows Sound Pack Error: {e}"


def ui_moss_reroll_single_event(
    scheme_name,
    theme_name,
    custom_style,
    ev_key,
    ev_custom_prompt,
    reroll_seed,
    pitch_semitones,
    steps,
    cfg_scale,
    sigma_shift,
    trim_db,
    fade_out_ms,
):
    try:
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
        dur = float(ev_meta["duration"])
        ev_prompt = (ev_custom_prompt or "").strip() or build_windows_event_prompt(ev_key, theme_name, custom_style)
        arr, sr, _, _ = MOSS_STUDIO.generate_one(
            prompt=ev_prompt,
            seconds=max(dur + 0.5, 1.5),
            num_inference_steps=int(steps),
            cfg_scale=float(cfg_scale),
            sigma_shift=float(sigma_shift),
            seed=int(reroll_seed),
        )
        mastered = master_ui_sound(
            arr, sr, target_duration_s=dur, trim_silence=True,
            trim_db=float(trim_db), fade_in_ms=1.5, fade_out_ms=float(fade_out_ms),
            pitch_semitones=float(pitch_semitones),
        )
        return finalize_rerolled_event_in_pack(
            ev_key=ev_key,
            mastered_wav=mastered,
            sr=sr,
            ev_prompt=ev_prompt,
            output_dir=OUTPUT_DIR,
            scheme_name=scheme_name,
            theme_name=theme_name,
            model_label="MOSS-SoundEffect-v2.0 (48 kHz)",
        )
    except Exception as e:
        return None, None, None, "", f"❌ Single-Event Re-Roll Error: {e}"


def ui_moss_batch_queue(batch_text, seconds, steps, cfg_scale, seed):
    try:
        lines = [ln.strip() for ln in (batch_text or "").splitlines() if ln.strip()]
        if not lines:
            return None, "❌ Enter at least one prompt line."
        saved_files = []
        base_seed = int(seed) if int(seed) >= 0 else 42
        for idx, line in enumerate(lines):
            arr, sr, _, _ = MOSS_STUDIO.generate_one(
                prompt=line, seconds=float(seconds), num_inference_steps=int(steps),
                cfg_scale=float(cfg_scale), seed=base_seed + idx,
            )
            arr = master_ui_sound(arr, sr, target_duration_s=float(seconds))
            f_out = os.path.join(OUTPUT_DIR, f"moss_batch_{idx+1:02d}_{int(time.time())}.wav")
            save_pcm16_wav(f_out, arr, sr)
            saved_files.append(f_out)
        return saved_files, f"✅ Batch generated **{len(saved_files)}** 48 kHz `.wav` files!"
    except Exception as e:
        return None, f"❌ Batch Error: {e}"


with gr.Blocks(title="MOSS-SoundEffect v2.0 — 48 kHz SFX Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## 🎧 MOSS-SoundEffect v2.0 — 48 kHz Flow-Matching DiT SFX & Windows Sound Pack Studio\n"
        "Powered by `OpenMOSS-Team/MOSS-SoundEffect-v2.0` (`1.3B` Flow-Matching DiT + `Qwen3-1.7B` + **48 kHz DAC VAE**). "
        "Supports **English & Chinese** prompts up to 30 seconds."
    )
    welcome_md = gr.Markdown("⏳ Initializing studio...")

    with gr.Tabs():
        with gr.Tab("🎧 1. 48 kHz Sound Effect Studio"):
            with gr.Row():
                with gr.Column(scale=3):
                    m_preset = gr.Dropdown(
                        choices=list(MOSS_PRESETS.keys()),
                        value=list(MOSS_PRESETS.keys())[0],
                        label="✨ Bilingual SFX Preset (EN / ZH)",
                        info="Select an English or Chinese preset to populate the prompt.",
                    )
                    m_prompt = gr.Textbox(
                        value=list(MOSS_PRESETS.values())[0],
                        lines=3,
                        label="📝 Sound Effect Prompt (English or Chinese)",
                        info="Describe the sound effect, material, action, and acoustic space.",
                    )
                    m_neg = gr.Textbox(
                        value="low quality, harsh noise, distortion, hum, clipping",
                        label="🚫 Negative Prompt",
                        info="Steers the Flow-Matching DiT away from unwanted background noise.",
                    )
                    with gr.Row():
                        m_sec = gr.Slider(1.0, 30.0, value=4.0, step=0.5, label="⏱️ Duration (Seconds)", info="Controllable duration from 1s up to 30s.")
                        m_steps = gr.Slider(20, 150, value=80, step=5, label="⚡ Flow-Matching Steps", info="Default 100 (or 50–80 for faster turnaround).")
                        m_cfg = gr.Slider(1.0, 8.0, value=4.0, step=0.5, label="🧭 CFG Scale", info="Recommended default: 4.0.")
                    with gr.Row():
                        m_shift = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label="📐 Sigma Shift", info="Flow-match scheduler shift (default: 5.0).")
                        m_seed = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = Random)", info="Set a fixed integer for reproducible audio.")
                        m_vars = gr.Slider(1, 2, value=2, step=1, label="📦 Variations", info="Generate 1 or 2 seed variations.")
                    with gr.Accordion("🛠️ Zero-Latency UI Mastering Controls", open=False):
                        with gr.Row():
                            m_master = gr.Checkbox(value=True, label="✨ Auto-Trim Leading Silence & Normalize", info="Removes dead air before first transient.")
                            m_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Silence Threshold (dB)", info="Pre-attack trim threshold.")
                            m_fade = gr.Slider(20.0, 400.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Smooth release tail.")
                    m_btn = gr.Button("🚀 Synthesize 48 kHz Sound Effect", variant="primary")
                with gr.Column(scale=2):
                    m_wav1 = gr.Audio(label="🎧 48 kHz Output (Take 1)", type="filepath")
                    m_wav2 = gr.Audio(label="🎧 48 kHz Output (Take 2)", type="filepath")
                    m_spec = gr.Image(label="📊 48 kHz Waveform & Spectrogram", type="filepath")
                    m_status = gr.Markdown()

            m_preset.change(fn=lambda k: MOSS_PRESETS.get(k, ""), inputs=[m_preset], outputs=[m_prompt])
            m_btn.click(
                fn=ui_moss_generate,
                inputs=[m_prompt, m_neg, m_sec, m_steps, m_cfg, m_shift, m_seed, m_vars, m_master, m_trim, m_fade],
                outputs=[m_wav1, m_wav2, m_spec, m_status],
            )

        with gr.Tab("🎚️ 2. 3-Layer Composite SFX & UI Mixer"):
            gr.Markdown(
                "### 🎚️ Professional 3-Layer Sound Effect Designer (48 kHz Stereo)\n"
                "Pro sound designers build iconic UI & game SFX by stacking **Layer 1 (Transient Click/Impact)** + "
                "**Layer 2 (Tonal Chime/Body)** + **Layer 3 (Shimmering Tail/Ambience)** with millisecond offsets and stereo panning."
            )
            with gr.Row():
                with gr.Column():
                    l1_p = gr.Textbox(value="Ultra-crisp short mechanical glass tap click transient", label="⚡ Layer 1: Attack / Transient Prompt", info="Immediate impact or click at 0ms.")
                    with gr.Row():
                        l1_d = gr.Slider(0.5, 10.0, value=1.0, step=0.5, label="Duration (s)", info="Layer 1 length.")
                        l1_off = gr.Slider(0, 1000, value=0, step=10, label="Offset (ms)", info="Start delay in ms.")
                        l1_g = gr.Slider(-18.0, 6.0, value=0.0, step=0.5, label="Gain (dB)", info="Layer 1 volume.")
                        l1_pan = gr.Slider(-1.0, 1.0, value=0.0, step=0.1, label="Pan (L/R)", info="-1 Left .. +1 Right.")

                    l2_p = gr.Textbox(value="Warm crystalline two-note ascending marimba chord chime", label="🎵 Layer 2: Core Tonal Body Prompt", info="Main melodic or timbral body.")
                    with gr.Row():
                        l2_d = gr.Slider(0.5, 10.0, value=2.0, step=0.5, label="Duration (s)", info="Layer 2 length.")
                        l2_off = gr.Slider(0, 1000, value=20, step=10, label="Offset (ms)", info="Start delay in ms.")
                        l2_g = gr.Slider(-18.0, 6.0, value=0.0, step=0.5, label="Gain (dB)", info="Layer 2 volume.")
                        l2_pan = gr.Slider(-1.0, 1.0, value=-0.15, step=0.05, label="Pan (L/R)", info="Slight stereo spread.")

                    l3_p = gr.Textbox(value="Soft airy celesta harmonic shimmer tail fade out", label="✨ Layer 3: Shimmer / Tail Prompt", info="Optional ambient tail or sub-bass layer.")
                    with gr.Row():
                        l3_d = gr.Slider(0.5, 10.0, value=2.5, step=0.5, label="Duration (s)", info="Layer 3 length.")
                        l3_off = gr.Slider(0, 1000, value=80, step=10, label="Offset (ms)", info="Start delay in ms.")
                        l3_g = gr.Slider(-18.0, 6.0, value=-4.0, step=0.5, label="Gain (dB)", info="Layer 3 volume.")
                        l3_pan = gr.Slider(-1.0, 1.0, value=0.20, step=0.05, label="Pan (L/R)", info="Slight stereo spread.")

                    with gr.Row():
                        mix_steps = gr.Slider(20, 120, value=60, step=5, label="⚡ Steps per Layer", info="Flow-matching steps.")
                        mix_cfg = gr.Slider(1.0, 8.0, value=4.0, step=0.5, label="🧭 CFG Scale", info="Guidance scale.")
                        mix_seed = gr.Number(value=101, precision=0, label="🎲 Base Seed", info="Seed.")
                    mix_btn = gr.Button("🎛️ Synthesize & Mix 3-Layer Stereo SFX", variant="primary")

                with gr.Column():
                    mix_out = gr.Audio(label="🎧 Master 48 kHz Stereo Composite SFX", type="filepath")
                    mix_spec = gr.Image(label="📊 Composite Stereo Spectrogram", type="filepath")
                    with gr.Accordion("🔍 Individual Layer Stems (48 kHz)", open=False):
                        stem1 = gr.Audio(label="Layer 1 Stem", type="filepath")
                        stem2 = gr.Audio(label="Layer 2 Stem", type="filepath")
                        stem3 = gr.Audio(label="Layer 3 Stem", type="filepath")
                    mix_status = gr.Markdown()

            mix_btn.click(
                fn=ui_moss_layer_mixer,
                inputs=[l1_p, l1_d, l1_off, l1_g, l1_pan, l2_p, l2_d, l2_off, l2_g, l2_pan, l3_p, l3_d, l3_off, l3_g, l3_pan, mix_steps, mix_cfg, mix_seed],
                outputs=[mix_out, stem1, stem2, stem3, mix_spec, mix_status],
            )

        with gr.Tab("🪟 3. 48 kHz Windows Sound Event Pack Builder"):
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Row():
                        mw_scheme = gr.Textbox(value="MOSS_Crystal_48k", label="🏷️ Windows Sound Scheme Name", info="Registered under HKCU\\AppEvents\\Schemes.")
                        mw_theme = gr.Dropdown(
                            choices=list(SONIC_THEME_PRESETS.keys()),
                            value=list(SONIC_THEME_PRESETS.keys())[0],
                            label="🎨 Sonic Theme Preset",
                            info="Select a cohesive theme for all Windows system events.",
                        )
                    mw_custom = gr.Textbox(value="", label="✍️ Custom Theme Style Override (Optional — EN or ZH)", info="Leave blank to use preset.")
                    mw_events = gr.CheckboxGroup(
                        choices=list(WINDOWS_SYSTEM_EVENTS.keys()),
                        value=DEFAULT_SELECTED_WIN_EVENTS,
                        label="🔔 Windows System Events to Include",
                        info="Each event is synthesized at native 48 kHz and mastered to 16-bit PCM.",
                    )
                    with gr.Row():
                        mw_steps = gr.Slider(25, 120, value=60, step=5, label="⚡ Steps per Event", info="60 steps balances speed & 48 kHz clarity.")
                        mw_cfg = gr.Slider(1.0, 8.0, value=4.0, step=0.5, label="🧭 CFG Scale", info="Default 4.0.")
                        mw_shift = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label="📐 Sigma Shift", info="Default 5.0.")
                        mw_seed = gr.Number(value=42, precision=0, label="🎲 Base Seed", info="Seed.")
                    with gr.Row():
                        mw_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Zero-Latency Trim (dB)", info="Removes pre-transient silence.")
                        mw_fade = gr.Slider(40.0, 300.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Click-free release.")
                    mw_btn = gr.Button("🪟 Build 48 kHz Windows Sound Scheme (.zip + .ps1 + .reg)", variant="primary")
                    with gr.Accordion("🎯 Single-Event Re-Roll & Pitch Customizer (Update 1 Event Without Regenerating Pack)", open=False):
                        with gr.Row():
                            mw_rr_ev = gr.Dropdown(choices=list(WINDOWS_SYSTEM_EVENTS.keys()), value="DeviceConnect", label="🔔 Event to Re-Roll / Replace", info="Select a single event to regenerate.")
                            mw_rr_seed = gr.Number(value=-1, precision=0, label="🎲 New Seed (-1 = Random)", info="Seed for this event.")
                            mw_rr_pitch = gr.Slider(-6.0, 6.0, value=0.0, step=0.5, label="🎵 Pitch Transpose (Semitones)", info="Semitone shift.")
                        mw_rr_prompt = gr.Textbox(value="", placeholder="Optional custom prompt override (EN or ZH)...", label="✍️ Single-Event Prompt Override (Optional)", info="Leave blank to use theme prompt.")
                        mw_rr_btn = gr.Button("🔄 Re-Roll Single Event & Update 48 kHz Pack")
                with gr.Column(scale=2):
                    mw_zip = gr.File(label="📦 Download 48 kHz Windows Sound Pack (.zip)")
                    mw_prev = gr.Audio(label="🎧 Sample Event Preview (48 kHz)", type="filepath")
                    mw_spec = gr.Image(label="📊 Spectrogram Check", type="filepath")
                    mw_status = gr.Markdown()
            mw_board = gr.HTML(label="🎹 Interactive 48 kHz Windows Sound Scheme Audition Board")

            mw_btn.click(
                fn=ui_moss_windows_pack,
                inputs=[mw_scheme, mw_theme, mw_custom, mw_events, mw_steps, mw_cfg, mw_shift, mw_seed, mw_trim, mw_fade],
                outputs=[mw_zip, mw_prev, mw_spec, mw_board, mw_status],
            )
            mw_rr_btn.click(
                fn=ui_moss_reroll_single_event,
                inputs=[mw_scheme, mw_theme, mw_custom, mw_rr_ev, mw_rr_prompt, mw_rr_seed, mw_rr_pitch, mw_steps, mw_cfg, mw_shift, mw_trim, mw_fade],
                outputs=[mw_zip, mw_prev, mw_spec, mw_board, mw_status],
            )

        with gr.Tab("📂 4. Batch Prompt Queue"):
            with gr.Row():
                with gr.Column():
                    b_text = gr.Textbox(
                        value=(
                            "Crisp wooden kalimba two-note ascending notification chime\n"
                            "Soft glass water drop pop sound effect\n"
                            "Futuristic sci-fi shield power down whoosh"
                        ),
                        lines=6,
                        label="📋 One Prompt per Line (EN or ZH)",
                        info="Each line is synthesized into a separate 48 kHz .wav file.",
                    )
                    with gr.Row():
                        b_sec = gr.Slider(1.0, 20.0, value=3.0, step=0.5, label="⏱️ Duration (s)", info="Duration per clip.")
                        b_steps = gr.Slider(25, 120, value=60, step=5, label="⚡ Steps", info="Flow-matching steps.")
                        b_cfg = gr.Slider(1.0, 8.0, value=4.0, step=0.5, label="🧭 CFG", info="Guidance scale.")
                        b_seed = gr.Number(value=42, precision=0, label="🎲 Seed", info="Base seed.")
                    b_btn = gr.Button("📦 Run Batch Queue", variant="primary")
                with gr.Column():
                    b_files = gr.File(label="📂 Generated 48 kHz Batch Files", file_count="multiple")
                    b_status = gr.Markdown()

            b_btn.click(fn=ui_moss_batch_queue, inputs=[b_text, b_sec, b_steps, b_cfg, b_seed], outputs=[b_files, b_status])

    demo.load(
        fn=lambda: f"🟢 **MOSS-SoundEffect v2.0 (48 kHz Flow-Matching DiT) Ready** on `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}`!",
        inputs=None,
        outputs=[welcome_md],
    )

clear_output()
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=False, show_error=True)

In [ ]:
#@title Step 5: Keep-Alive Helper & GPU VRAM Monitor
import time
import torch

KEEP_ALIVE_MINUTES = 0 #@param {type:"integer"}

try:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(0) / (1024 ** 3)
        resv = torch.cuda.memory_reserved(0) / (1024 ** 3)
        print(f"📊 GPU VRAM Status — Allocated: {alloc:.2f} GB | Reserved: {resv:.2f} GB")
    if KEEP_ALIVE_MINUTES > 0:
        print(f"⏳ Keeping Gradio session alive for {KEEP_ALIVE_MINUTES} minutes...")
        for m in range(KEEP_ALIVE_MINUTES):
            time.sleep(60)
            print(f"  ✓ Active: {m + 1}/{KEEP_ALIVE_MINUTES} min elapsed")
    else:
        print("ℹ️ Keep-alive loop skipped (KEEP_ALIVE_MINUTES=0). Set > 0 if running unattended.")
except KeyboardInterrupt:
    print("⏹️ Keep-alive stopped by user.")
except Exception as e:
    print(f"Keep-alive warning: {e}")

In [ ]:
#@title Step 6: Quick Test — Single 48 kHz Sound Effect
import os
from IPython.display import Audio, FileLink, display

QUICKTEST_PROMPT = "A crisp crystalline glass marimba two-note ascending notification chime with a soft airy tail." #@param {type:"string"}
QUICKTEST_SECONDS = 3.0 #@param {type:"number"}
QUICKTEST_SEED = 42 #@param {type:"integer"}

try:
    wav_path, _, _, msg = ui_moss_generate(
        prompt=QUICKTEST_PROMPT,
        negative_prompt="low quality, noise, distortion",
        seconds=QUICKTEST_SECONDS,
        steps=60,
        cfg_scale=4.0,
        sigma_shift=5.0,
        seed=QUICKTEST_SEED,
        num_variations=1,
        apply_mastering=True,
        trim_db=-42.0,
        fade_out_ms=120.0,
    )
    print(msg)
    if wav_path and os.path.exists(wav_path):
        display(Audio(filename=wav_path))
        display(FileLink(wav_path))
except Exception as e:
    print(f"Quick test error: {e}")

In [ ]:
#@title Step 7: Headless Batch 48 kHz Windows Sound Pack Export
import os
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_SCHEME_NAME = "MOSS_Crystal_48k" #@param {type:"string"}
BATCH_THEME = "Frutiger Aero Crystal Glass" #@param ["Frutiger Aero Crystal Glass", "Cyberpunk Neon HUD", "Cozy Wooden Kalimba & Mallet", "Sci-Fi Spaceship Bridge", "Retro 16-Bit Chiptune Arcade", "Warm Lo-Fi Analog Tape Synth", "Minimalist Tactile Micro-Foley", "Dark Fantasy Crystal & Harp"]
BATCH_SEED = 42 #@param {type:"integer"}

try:
    event_map = {}
    pack_dir = os.path.join(OUTPUT_DIR, f"batch_moss_winpack_{BATCH_SEED}")
    os.makedirs(pack_dir, exist_ok=True)
    for idx, ev_key in enumerate(tqdm(DEFAULT_SELECTED_WIN_EVENTS, desc="MOSS 48 kHz Windows Pack")):
        try:
            ev_meta = WINDOWS_SYSTEM_EVENTS[ev_key]
            prompt = build_windows_event_prompt(ev_key, BATCH_THEME, "")
            arr, sr, _, _ = MOSS_STUDIO.generate_one(
                prompt=prompt,
                seconds=max(float(ev_meta["duration"]) + 0.5, 1.5),
                num_inference_steps=60,
                cfg_scale=4.0,
                sigma_shift=5.0,
                seed=BATCH_SEED + idx,
            )
            mastered = master_ui_sound(arr, sr, target_duration_s=float(ev_meta["duration"]))
            fpath = os.path.join(pack_dir, f"{ev_key}.wav")
            save_pcm16_wav(fpath, mastered, sr)
            event_map[ev_key] = {"path": fpath, "filename": f"{ev_key}.wav", "prompt": prompt}
        except Exception as item_e:
            print(f"⚠️ Event {ev_key} skipped: {item_e}")

    if event_map:
        zip_out = package_windows_sound_scheme_zip(BATCH_SCHEME_NAME, event_map, BATCH_THEME, "MOSS-SoundEffect-v2.0 (48 kHz)", OUTPUT_DIR)
        print(f"✅ Packaged 48 kHz Windows Sound Scheme: {zip_out}")
        display(FileLink(zip_out))
except Exception as e:
    print(f"Batch Windows Pack error: {e}")